# EasyMLOps  
  
## 介绍   
`EasyMLOps`包以`Pipline`的方式构建建模任务，可直接进行模型训练、预测(离线，在线)，测试(离线在线预测一致性、预测性能)、特征存储、监控、分析等功能，通过外套一层Flask或FastApi即可直接部署生产

## 安装
```bash
pip install easymlops-版本号
```  

## 使用

In [1]:
import pandas as pd
data=pd.read_csv("./data/demo.csv")
print(data.head(5).to_markdown())

|    |   PassengerId |   Survived |   Pclass | Name                                                | Sex    |   Age |   SibSp |   Parch | Ticket           |    Fare | Cabin   | Embarked   |
|---:|--------------:|-----------:|---------:|:----------------------------------------------------|:-------|------:|--------:|--------:|:-----------------|--------:|:--------|:-----------|
|  0 |             1 |          0 |        3 | Braund, Mr. Owen Harris                             | male   |    22 |       1 |       0 | A/5 21171        |  7.25   | nan     | S          |
|  1 |             2 |          1 |        1 | Cumings, Mrs. John Bradley (Florence Briggs Thayer) | female |    38 |       1 |       0 | PC 17599         | 71.2833 | C85     | C          |
|  2 |             3 |          1 |        3 | Heikkinen, Miss. Laina                              | female |    26 |       0 |       0 | STON/O2. 3101282 |  7.925  | nan     | S          |
|  3 |             4 |          1 |        1 | Fut

In [2]:
# 拆分训练测试
x_train=data[:500]
x_test=data[500:]
y_train=x_train["Survived"]
y_test=x_test["Survived"]
del x_train["Survived"]
del x_test["Survived"]

### 表格型任务

In [3]:
from easymlops import TablePipeLine
from easymlops.table.strategy import LGBMRegressionLayers
from easymlops.table.preprocessing import *
from easymlops.table.encoding import *
from easymlops.table.decomposition import *
from easymlops.table.classification import *
from easymlops.table.ensemble import *

In [4]:
x_train

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...
495,496,3,"Yousseff, Mr. Gerious",male,NaN,0,0,2627,14.4583,NaN,C
496,497,1,"Eustis, Miss. Elizabeth Mussey",female,54.0,1,0,36947,78.2667,D20,C
497,498,3,"Shellard, Mr. Frederick William",male,NaN,0,0,C.A. 6212,15.1000,NaN,S
498,499,1,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.0,1,2,113781,151.5500,C22 C26,S


In [5]:
table = TablePipeLine()
table.pipe(FixInput()) \
.pipe(FillNa()) \
.pipe(TargetEncoding(y=y_train,cols=["Name","Sex","Ticket","Cabin","Embarked"]))\
.pipe(Normalizer())\
.pipe(TCADecomposition(dim=10))

table.fit(x_train)

(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Name ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Ticket ,the std is 0,the finally result will be set 1


In [6]:
table.transform(x_test)

(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Name ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Ticket ,the std is 0,the finally result will be set 1


,0,1,2,3,4,5,6,7,8,9
500,-0.100830,-0.154102,0.083117,-0.098807,-0.137425,-0.105033,-0.007652,0.017110,-0.000508,0.233336
501,0.012091,-0.067595,0.114583,-0.337780,-0.390594,0.089951,-0.007703,0.013382,-0.040349,0.238596
502,0.008659,-0.032970,0.114915,-0.312214,-0.363795,-0.078512,-0.002218,-0.028031,-0.042386,0.224301
503,-0.009362,-0.146483,-0.056311,-0.158405,-0.386739,-0.085190,-0.014571,0.065191,0.012083,0.221133
504,0.150107,-0.089159,-0.013999,-0.092828,-0.237668,-0.417815,-0.009422,0.024445,0.037291,0.244262
...,...,...,...,...,...,...,...,...,...,...
886,0.005209,-0.203423,0.089011,-0.137912,-0.193487,0.044804,-0.025703,-0.100506,-0.051300,0.130070
887,0.118805,-0.159938,0.027407,-0.147433,-0.289932,-0.241189,-0.005984,0.073645,-0.086758,0.150394
888,0.043279,-0.116605,-0.077012,-0.010162,-0.195411,0.033236,0.006140,-0.067612,-0.042517,0.165972
889,0.082031,-0.155968,0.193514,-0.223550,-0.168811,0.065369,-0.007967,0.068938,-0.113220,0.168107


In [7]:
table.transform_single(x_test.to_dict("record")[0])

(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Name ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Ticket ,the std is 0,the finally result will be set 1


{0: -0.10083013037160878,
 1: -0.15410229957220328,
 2: 0.08311711977385502,
 3: -0.09880650440979628,
 4: -0.13742528591919376,
 5: -0.10503307395080759,
 6: -0.007651860389966335,
 7: 0.017109957126614103,
 8: -0.0005080848921934815,
 9: 0.233335699951911}

In [8]:
# table.auto_test(x_test[:10])

In [9]:
table.save("demo.pkl")

In [10]:
new_table=TablePipeLine()
new_table.load("demo.pkl")

In [11]:
print(new_table.transform(x_test).head(5).to_markdown())

(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Name ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Ticket ,the std is 0,the finally result will be set 1
|     |           0 |          1 |          2 |          3 |         4 |          5 |           6 |          7 |            8 |        9 |
|----:|------------:|-----------:|-----------:|-----------:|----------:|-----------:|------------:|-----------:|-------------:|---------:|
| 500 | -0.10083    | -0.154102  |  0.0831171 | -0.0988065 | -0.137425 | -0.105033  | -0.00765186 |  0.01711   | -0.000508085 | 0.233336 |
| 501 |  0.0120915  | -0.0675953 |  0.114583  | -0.33778   | -0.390594 |  0.0899509 | -0.00770269 |  0.0133823 | -0.0403488   | 0.238596 |
| 502 |  0.00865851 | -0.0329695 |  0.114915  | -0.312214  | -0.363795 | -0.0785117 | -0.00221762 | -0.0280313 | -0.0423861   | 0.224301 |
| 503 | -0.0093624  | -0.146483  | 

In [12]:
new_table.transform_single(x_test.to_dict("record")[0])

(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Name ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Ticket ,the std is 0,the finally result will be set 1


{0: -0.10083013037160878,
 1: -0.15410229957220328,
 2: 0.08311711977385502,
 3: -0.09880650440979628,
 4: -0.13742528591919376,
 5: -0.10503307395080759,
 6: -0.007651860389966335,
 7: 0.017109957126614103,
 8: -0.0005080848921934815,
 9: 0.233335699951911}

In [14]:
new_table.auto_test(x_test[:10])


###################################################################
 1.一致性测试和性能测试:check_transform_function                      
###################################################################
(<class 'easymlops.table.preprocessing.core.FixInput'>) module check [transform] complete,speed:[0.0ms]/it,cpu:[0%],memory:[0K]
(<class 'easymlops.table.preprocessing.onevar_operation.FillNa'>) module check [transform] complete,speed:[0.0ms]/it,cpu:[0%],memory:[0K]
(<class 'easymlops.table.encoding.TargetEncoding'>) module check [transform] complete,speed:[0.0ms]/it,cpu:[0%],memory:[0K]
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Name ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Ticket ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Name ,the std is 0,the finally result will be set 1
(<class 

(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Name ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Ticket ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Name ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Ticket ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Name ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Ticket ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Name ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.

(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Name ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Ticket ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Name ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Ticket ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Name ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Ticket ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Name ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.

(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Name ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Ticket ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Name ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Ticket ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Name ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Ticket ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Name ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.

(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Name ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Ticket ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Name ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Ticket ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Name ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Ticket ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Name ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.

...

(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Name ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Ticket ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Name ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Ticket ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Name ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Ticket ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Name ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.

(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Name ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Ticket ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Name ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Ticket ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Name ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Ticket ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Name ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.

(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Name ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Ticket ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Name ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Ticket ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Name ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Ticket ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Name ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.

(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Name ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Ticket ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Name ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Ticket ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Name ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Ticket ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Name ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.

(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Name ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Ticket ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Name ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Ticket ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Name ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Ticket ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>), in  column Name ,the std is 0,the finally result will be set 1
(<class 'easymlops.table.preprocessing.onevar_operation.